nlu.yml

In [ ]:
version: "3.1"
nlu:
  - intent: saludar
    examples: |
      - hola
      - buenos días
      - qué tal
      - buenas tardes

  - intent: consultar_homicidios_dolosos
    examples: |
      - ¿Cuántos [homicidios dolosos](delito) hubo en [2023](periodo) en [Iztapalapa](alcaldia)?
      - ¿Cuántos [femin icidios](delito) hubo en [2023](periodo) en [Iztacalco](alcaldia)?
      - ¿Cuántos [homicidios dolosos](delito) hubo el [mes pasado](periodo) en [Iztapalapa](alcaldia)?
      - Dime cuántos [homicidio por arma de fuego](delito) hubo en [2024](periodo) en [Cuauhtémoc](alcaldia).
      - ¿Cuántas carpetas de [feminicidio por golpes](delito) se abrieron en [2023](periodo)?
      - ¿Cuántas víctimas de [homicidios dolosos](delito) hubo en [Iztapalapa](alcaldia) en [2023](periodo)?
      - ¿Cuántos [homicidios dolosos](delito) hubo en [Alvaro Obregon](alcaldia) en [2023](periodo)?
      - ¿Cuántos [homicidios dolosos](delito) hubo en [2024](periodo) en [Alvaro Obregon](alcaldia)?
      - Dime cuántos [homicidios por golpes](delito) ocurrieron en [Alvaro Obregon](alcaldia) durante [2022](periodo)
      - ¿Cuantos [homicidios dolosos](delito) hubo en [2024](periodo) en [Álvaro Obregón](alcaldia)?

  - intent: informar_alcaldia
    examples: |
      - En [Iztapalapa](alcaldia)
      - La alcaldía es [Cuauhtémoc](alcaldia)
      - Es en [Álvaro Obregón](alcaldia)
      - La alcaldia es [Cuauhtemoc](alcaldia)
      - En [Alvaro Obregon](alcaldia)
      - [Alvaro Obregon](alcaldia)
      - [Álvaro Obregón](alcaldia)

  - intent: informar_delito
    examples: |
      - El delito es [homicidios dolosos](delito)
      - Busco [feminicidio por arma blanca](delito)
      - Es sobre [homicidio por arma de fuego](delito)

  - intent: informar_periodo
    examples: |
      - El año es [2023](periodo)T
      - En [2024](periodo)
      - [Mes pasado](periodo)

  - lookup: alcaldia
    examples: |
      - Álvaro Obregón
      - Alvaro Obregon
      - Azcapotzalco
      - Benito Juárez
      - Benito Juarez
      - Coyoacán
      - Coyoacan
      - Cuajimalpa de Morelos
      - Cuauhtémoc
      - Cuauhtemoc
      - Gustavo A. Madero
      - Gustavo A Madero
      - Iztacalco
      - Iztapalapa
      - La Magdalena Contreras
      - Miguel Hidalgo
      - Milpa Alta
      - Tláhuac
      - Tlahuac
      - Tlalpan
      - Venustiano Carranza
      - Xochimilco

  - lookup: delito
    examples: |
      - homicidios dolosos
      - feminicidio
      - feminicidio por arma blanca
      - feminicidio por disparo de arma de fuego
      - feminicidio por golpes
      - homicidio intencional y robo de vehiculo
      - homicidio por ahorcamiento
      - homicidio por arma blanca
      - homicidio por arma de fuego
      - homicidio por golpes
      - homicidios intencionales (otros)

config.yml

In [ ]:
version: "3.1"

recipe: default.v1
assistant_id: 20250223-135104-old-mast

language: es

pipeline:
  - name: SpacyNLP
    model: "es_core_news_sm"
  - name: SpacyTokenizer
  - name: SpacyFeaturizer
  - name: RegexFeaturizer
  - name: CountVectorsFeaturizer
  - name: DIETClassifier
    epochs: 100
    model_confidence: softmax
    constrain_similarities: true
  - name: EntitySynonymMapper
  - name: ResponseSelector
    epochs: 30
    model_confidence: softmax
    constrain_similarities: true
  - name: FallbackClassifier
    threshold: 0.5
    ambiguity_threshold: 0.1

policies:
  - name: RulePolicy
    core_fallback_threshold: 0.3
    core_fallback_action_name: action_default_fallback
    enable_fallback_prediction: true
  - name: TEDPolicy
    max_history: 5
    epochs: 100
    model_confidence: softmax
    constrain_similarities: true

domain.yml

In [ ]:
version: "3.1"
language: es

intents:
  - saludar
  - consultar_homicidios_dolosos
  - informar_alcaldia
  - informar_delito
  - informar_periodo

entities:
  - periodo
  - alcaldia
  - delito

slots:
  periodo:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: periodo
  alcaldia:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: alcaldia
  delito:
    type: text
    influence_conversation: true
    mappings:
      - type: from_entity
        entity: delito

actions:
  - action_consultar_homicidios_dolosos
  - utter_greet
  - utter_ask_periodo
  - utter_ask_alcaldia
  - utter_ask_delito

responses:
  utter_greet:
    - text: "¡Hola! ¿En qué puedo ayudarte?"
  utter_ask_periodo:
    - text: "Por favor, dime el período (ej. '2023' o 'mes pasado')."
  utter_ask_alcaldia:
    - text: "Por favor, especifica una alcaldía (ej. 'Iztapalapa')."
  utter_ask_delito:
    - text: "Por favor, dime el delito (ej. 'homicidios dolosos' o 'feminicidio')."

session_config:
  session_expiration_time: 60
  carry_over_slots_to_new_session: true

stories.yml

In [ ]:
'version: "3.1"

stories:
  - story: Saludo simple
    steps:
      - intent: saludar
      - action: utter_greet

  - story: Consulta completa
    steps:
      - intent: consultar_homicidios_dolosos
        entities:
          - periodo: "2023"
          - alcaldia: "Iztapalapa"
          - delito: "homicidios dolosos"
      - action: action_consultar_homicidios_dolosos

  - story: Consulta con mes pasado
    steps:
      - intent: consultar_homicidios_dolosos
        entities:
          - periodo: "mes pasado"
          - alcaldia: "Iztapalapa"
          - delito: "homicidios dolosos"
      - action: action_consultar_homicidios_dolosos

  - story: Consulta sin alcaldía
    steps:
      - intent: consultar_homicidios_dolosos
        entities:
          - periodo: "2023"
          - delito: "feminicidio"
      - action: utter_ask_alcaldia
      - intent: informar_alcaldia
        entities:
          - alcaldia: "Iztapalapa"
      - action: action_consultar_homicidios_dolosos

credentials.yml

In [ ]:
rest:
#  # you don't need to provide anything here - this channel doesn't
#  # require any credentials

telegram:
  access_token: "7662937524:AAEL-8PhXIIoRevDgtsCKSdttiue9ps8Blw"
  verify: "ADIP_interno_Bot"
  webhook_url: "https://7984-148-208-135-2.ngrok-free.app/webhooks/telegram/webhook"

#facebook:
#  verify: "<verify>"
#  secret: "<your secret>"
#  page-access-token: "<your page access token>"

#slack:
#  slack_token: "<your slack token>"
#  slack_channel: "<the slack channel>"
#  slack_signing_secret: "<your slack signing secret>"

#socketio:
#  user_message_evt: <event name for user message>
#  bot_message_evt: <event name for bot messages>
#  session_persistence: <true/false>

#mattermost:
#  url: "https://<mattermost instance>/api/v4"
#  token: "<bot token>"
#  webhook_url: "<callback URL>"

# This entry is needed if you are using Rasa Enterprise. The entry represents credentials
# for the Rasa Enterprise "channel", i.e. Talk to your bot and Share with guest testers.

rasa:
  url: "http://localhost:5005/api"

actions.py

In [ ]:
from typing import Any, Text, Dict, List
from rasa_sdk import Action, Tracker
from rasa_sdk.executor import CollectingDispatcher
import pandas as pd
import os
import re
import unicodedata
import chardet
import io
from datetime import datetime, timedelta
import difflib  # Para fuzzy matching en alcaldías

def normalize_text(text: str) -> str:
    """Normaliza un texto eliminando acentos y convirtiéndolo a minúsculas."""
    if not isinstance(text, str):
        return ""
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8').lower().strip()

def detect_encoding(file_path: str, num_bytes: int = 10000) -> str:
    """
    Detecta la codificación del archivo leyendo los primeros num_bytes.
    """
    with open(file_path, 'rb') as f:
        rawdata = f.read(num_bytes)
    result = chardet.detect(rawdata)
    encoding = result.get('encoding', 'utf-8')
    print(f"Encoding detectado: {encoding}")
    return encoding

# Función para normalizar delitos (añadida de la segunda versión)
def normalizar_delito(delito: str) -> str:
    """Normaliza texto de delitos para comparación consistente."""
    if not isinstance(delito, str):
        return ""
    return normalize_text(delito)

# --- Carga global de la base de datos ---
DATA_PATH = os.path.join("data", "df.csv")
try:
    # Detectar la codificación
    print(f"Intentando cargar archivo desde: {DATA_PATH}")
    encoding_detected = detect_encoding(DATA_PATH)

    # Leer el archivo con la codificación detectada
    with open(DATA_PATH, 'rb') as f:
        rawdata = f.read()

    # Decodificar el contenido
    text = rawdata.decode(encoding_detected, errors='replace')

    # Usar StringIO para pasar el contenido decodificado a pd.read_csv
    df_global = pd.read_csv(io.StringIO(text))

    # Imprimir información del DataFrame para depuración
    print("Base de datos cargada correctamente.")
    print(df_global.head())
    print(f"Columnas en el DataFrame: {list(df_global.columns)}")

    # Convertir la columna de fechas al formato correcto
    if 'fecha' in df_global.columns:
        df_global['fecha'] = pd.to_datetime(
            df_global['fecha'],
            dayfirst=True,  # Importante para formato dd/mm/yyyy
            errors='coerce'
        )

    # Asegurarse de que la columna año es de tipo numérico
    if 'aÒo' in df_global.columns:
        df_global.rename(columns={'aÒo': 'anio'}, inplace=True)
        df_global['anio'] = pd.to_numeric(df_global['anio'], errors='coerce')
    elif 'año' in df_global.columns:
        df_global.rename(columns={'año': 'anio'}, inplace=True)
        df_global['anio'] = pd.to_numeric(df_global['anio'], errors='coerce')

except FileNotFoundError:
    print(f"Error: No se encontró el archivo en {DATA_PATH}")
    df_global = None
except Exception as e:
    print(f"Error al cargar la base de datos: {e}")
    df_global = None

class ActionConsultarHomicidiosDolosos(Action):
    def name(self) -> Text:
        return "action_consultar_homicidios_dolosos"

    def run(self, dispatcher: CollectingDispatcher,
            tracker: Tracker,
            domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:

        if df_global is None:
            dispatcher.utter_message(text="Lo siento, no puedo acceder a la base de datos en este momento.")
            return []

        # Obtener el mensaje original del usuario para análisis de patrones
        last_user_message = tracker.latest_message.get('text', '').lower()
        print(f"Mensaje del usuario: {last_user_message}")

        # Extraer entidades
        alcaldia = next(tracker.get_latest_entity_values("alcaldia"), None)
        periodo = next(tracker.get_latest_entity_values("periodo"), None)
        delito = next(tracker.get_latest_entity_values("delito"), None)

        # Verificar en los slots si no se encontraron entidades
        if alcaldia is None:
            alcaldia = tracker.get_slot("alcaldia")

        if periodo is None:
            periodo = tracker.get_slot("periodo")

        if delito is None:
            delito = tracker.get_slot("delito")

        print(f"Entidades extraídas: alcaldia={alcaldia}, periodo={periodo}, delito={delito}")

        # PASO 1: DETERMINAR EL TIPO DE DELITO EXACTO
        # Esta es la parte crítica para garantizar respuestas correctas
        tipo_delito_mensaje = "homicidios dolosos"  # Valor por defecto
        delito_exacto = None

        # Analizar el mensaje completo para identificar el tipo específico de delito
        if "feminicidio" in last_user_message:
            if "arma blanca" in last_user_message:
                tipo_delito_mensaje = "feminicidios por arma blanca"
                delito_exacto = "FEMINICIDIO POR ARMA BLANCA"
            elif "arma de fuego" in last_user_message:
                tipo_delito_mensaje = "feminicidios por arma de fuego"
                delito_exacto = "FEMINICIDIO POR ARMA DE FUEGO"
            else:
                tipo_delito_mensaje = "feminicidios"
                delito_exacto = "FEMINICIDIO"
        elif "homicidio" in last_user_message or "homicidios" in last_user_message:
            if "arma blanca" in last_user_message:
                tipo_delito_mensaje = "homicidios por arma blanca"
                delito_exacto = "HOMICIDIO POR ARMA BLANCA"
            elif "arma de fuego" in last_user_message:
                tipo_delito_mensaje = "homicidios por arma de fuego"
                delito_exacto = "HOMICIDIO POR ARMA DE FUEGO"
            elif "golpes" in last_user_message:
                tipo_delito_mensaje = "homicidios por golpes"
                delito_exacto = "HOMICIDIO POR GOLPES"
            else:
                tipo_delito_mensaje = "homicidios dolosos"

        # También analizar la entidad delito si fue detectada
        if delito:
            delito_lower = delito.lower()
            if "feminicidio" in delito_lower:
                if "arma blanca" in delito_lower:
                    tipo_delito_mensaje = "feminicidios por arma blanca"
                    delito_exacto = "FEMINICIDIO POR ARMA BLANCA"
                elif "arma de fuego" in delito_lower:
                    tipo_delito_mensaje = "feminicidios por arma de fuego"
                    delito_exacto = "FEMINICIDIO POR ARMA DE FUEGO"
                else:
                    tipo_delito_mensaje = "feminicidios"
                    delito_exacto = "FEMINICIDIO"
            elif "arma blanca" in delito_lower:
                tipo_delito_mensaje = "homicidios por arma blanca"
                delito_exacto = "HOMICIDIO POR ARMA BLANCA"
            elif "arma de fuego" in delito_lower:
                tipo_delito_mensaje = "homicidios por arma de fuego"
                delito_exacto = "HOMICIDIO POR ARMA DE FUEGO"
            elif "golpes" in delito_lower:
                tipo_delito_mensaje = "homicidios por golpes"
                delito_exacto = "HOMICIDIO POR GOLPES"

        print(f"Tipo de delito identificado: {tipo_delito_mensaje} (DB: {delito_exacto})")

        # Si no se proporciona alcaldía, consultar todas
        todas_alcaldias = alcaldia is None

        # Inicializar alcaldia_para_mensaje con un valor predeterminado
        alcaldia_para_mensaje = "toda la Ciudad de México"

        # Crear una copia del DataFrame para no modificar el original
        df = df_global.copy()

        # PASO 2: FILTRAR POR EL TIPO DE DELITO EXACTO
        # Si tenemos un delito exacto identificado, usarlo para filtrar
        if delito_exacto and 'delito' in df.columns:
            # Método 1: Intentar coincidencia exacta primero
            exact_matches = df[df['delito'] == delito_exacto]
            if len(exact_matches) > 0:
                print(f"Encontrados {len(exact_matches)} registros con coincidencia exacta para {delito_exacto}")
                df = exact_matches
            else:
                # Método 2: Usar coincidencia parcial
                partial_matches = df[df['delito'].apply(
                    lambda x: delito_exacto.lower() in x.lower() if isinstance(x, str) else False
                )]
                if len(partial_matches) > 0:
                    print(f"Encontrados {len(partial_matches)} registros con coincidencia parcial para {delito_exacto}")
                    df = partial_matches
                else:
                    # Método 3: Buscar por términos clave
                    terms = [term.lower() for term in delito_exacto.split() if len(term) > 3]
                    print(f"Buscando términos: {terms}")

                    filtered_df = df.copy()
                    for term in terms:
                        filtered_df = filtered_df[filtered_df['delito'].apply(
                            lambda x: term in x.lower() if isinstance(x, str) else False
                        )]

                    if len(filtered_df) > 0:
                        print(f"Encontrados {len(filtered_df)} registros por términos clave")
                        df = filtered_df
        else:
            # Si no tenemos un delito exacto, usar la lógica original
            if 'categoria' in df.columns:
                df = df[df['categoria'].apply(
                    lambda x: 'homicidio doloso' in normalize_text(x) if isinstance(x, str) else False
                )]

        # PASO 3: FILTRAR POR AÑO/PERIODO
        if periodo is not None:
            try:
                año = int(periodo)
                if 'anio' in df.columns:
                    df = df[df['anio'] == año]
                elif 'fecha' in df.columns:
                    df = df[df['fecha'].dt.year == año]
                print(f"Después de filtrar por año {año} quedan {len(df)} registros")
            except ValueError:
                print(f"Periodo no numérico: {periodo}")

        # PASO 4: FILTRAR POR ALCALDÍA
        alcaldia_encontrada = False
        if not todas_alcaldias and 'alcaldias' in df.columns:
            alcaldia_norm = normalize_text(alcaldia)

            # Filtrar por nombre normalizado de alcaldía
            alcaldias_dict = {
                "iztacalco": "IZTACALCO",
                "iztapalapa": "IZTAPALAPA",
                "alvaro obregon": "ALVARO OBREGON",
                "cuauhtemoc": "CUAUHTEMOC",
                "gustavo a madero": "GUSTAVO A MADERO",
                "tlahuac": "TLAHUAC",
                "tlalpan": "TLALPAN"
            }

            # Verificar cada clave conocida
            for key, value in alcaldias_dict.items():
                if key in alcaldia_norm:
                    filtered_df = df[df['alcaldias'].apply(
                        lambda x: normalize_text(x) == key if isinstance(x, str) else False
                    )]
                    if len(filtered_df) > 0:
                        df = filtered_df
                        alcaldia_para_mensaje = value
                        alcaldia_encontrada = True
                        break

            if not alcaldia_encontrada:
                # Intento adicional con todas las alcaldías disponibles
                alcaldias_unicas = df['alcaldias'].dropna().unique()
                alcaldias_unicas_norm = [normalize_text(alc) if isinstance(alc, str) else "" for alc in alcaldias_unicas]

                # Intento de coincidencia exacta
                if alcaldia_norm in alcaldias_unicas_norm:
                    idx = alcaldias_unicas_norm.index(alcaldia_norm)
                    alcaldia_match = alcaldias_unicas[idx]
                    df = df[df['alcaldias'].apply(lambda x: normalize_text(x) == alcaldia_norm if isinstance(x, str) else False)]
                    alcaldia_para_mensaje = alcaldia_match
                    alcaldia_encontrada = True
                else:
                    # Intento de coincidencia aproximada
                    matches = difflib.get_close_matches(alcaldia_norm, alcaldias_unicas_norm, n=1, cutoff=0.8)
                    if matches:
                        idx = alcaldias_unicas_norm.index(matches[0])
                        alcaldia_match = alcaldias_unicas[idx]
                        df = df[df['alcaldias'].apply(lambda x: normalize_text(x) == matches[0] if isinstance(x, str) else False)]
                        alcaldia_para_mensaje = alcaldia_match
                        alcaldia_encontrada = True

        # PASO 5: CONSTRUIR LA RESPUESTA
        total_delitos = len(df)
        print(f"Total de delitos encontrados después de filtrar: {total_delitos}")

        # Si se especificó una alcaldía pero no se encontró, dar un mensaje informativo
        if not todas_alcaldias and not alcaldia_encontrada:
            mensaje = f"No se encontraron registros de {tipo_delito_mensaje} para la alcaldía {alcaldia}"
            if periodo is not None:
                mensaje += f" durante {periodo}."
            else:
                mensaje += "."
            dispatcher.utter_message(text=mensaje)
            return []

        # Construir mensaje final
        if todas_alcaldias:
            mensaje = f"Se registraron {total_delitos} {tipo_delito_mensaje} en toda la Ciudad de México"
        else:
            mensaje = f"Se registraron {total_delitos} {tipo_delito_mensaje} en la alcaldía {alcaldia_para_mensaje}"

        if periodo is not None:
            mensaje += f" durante {periodo}."
        else:
            mensaje += "."

        dispatcher.utter_message(text=mensaje)

        return []

class ActionConsultarDelitoPorFecha(Action):
    def name(self) -> Text:
        return "action_consultar_delito_por_fecha"

    def run(self, dispatcher: CollectingDispatcher,
            tracker: Tracker,
            domain: Dict[Text, Any]) -> List[Dict[Text, Any]]:

        if df_global is None:
            dispatcher.utter_message(text="Lo siento, no puedo acceder a la base de datos en este momento.")
            return []

        # Obtener el mensaje original del usuario
        last_user_message = tracker.latest_message.get('text', '').lower()
        print(f"Mensaje del usuario: {last_user_message}")

        # Extraer entidades
        delito = next(tracker.get_latest_entity_values("delito"), None)
        alcaldia = next(tracker.get_latest_entity_values("alcaldia"), None)
        periodo = next(tracker.get_latest_entity_values("periodo"), None)

        # Verificar en los slots si no se encontraron entidades
        if delito is None:
            delito = tracker.get_slot("delito")

        if alcaldia is None:
            alcaldia = tracker.get_slot("alcaldia")

        if periodo is None:
            periodo = tracker.get_slot("periodo")

        print(f"Entidades extraídas: alcaldia={alcaldia}, periodo={periodo}, delito={delito}")

        # Verificar si se proporcionó un delito
        if delito is None:
            dispatcher.utter_message(text="Por favor, especifica el tipo de delito que deseas consultar.")
            return []

        # Determinar el tipo de delito para el mensaje y la búsqueda
        tipo_delito_mensaje = delito.lower()
        delito_norm = normalize_text(delito)

        # Si no se proporciona alcaldía, consultar todas
        todas_alcaldias = alcaldia is None

        # Inicializar alcaldia_para_mensaje con un valor predeterminado
        alcaldia_para_mensaje = "toda la Ciudad de México"

        # Crear una copia del DataFrame para no modificar el original
        df = df_global.copy()

        # Filtrar por delito de manera más flexible
        if 'delito' in df.columns:
            # Primero intenta una coincidencia exacta
            exact_matches = df[df['delito'].apply(
                lambda x: delito_norm == normalize_text(x) if isinstance(x, str) else False
            )]

            if len(exact_matches) > 0:
                df = exact_matches
            else:
                # Si no hay exacta, busca coincidencias parciales
                words = delito_norm.split()
                if len(words) > 1:
                    # Si hay varias palabras, buscar coincidencias de las palabras más significativas
                    significant_words = [w for w in words if len(w) > 3]  # Palabras de más de 3 letras

                    filtered_df = df.copy()
                    for word in significant_words:
                        filtered_df = filtered_df[filtered_df['delito'].apply(
                            lambda x: word in normalize_text(x) if isinstance(x, str) else False
                        )]

                    if len(filtered_df) > 0:
                        df = filtered_df
                else:
                    # Si es una sola palabra, buscar coincidencia directa
                    df = df[df['delito'].apply(
                        lambda x: delito_norm in normalize_text(x) if isinstance(x, str) else False
                    )]

        # Filtrar por año/periodo
        if periodo is not None:
            try:
                año = int(periodo)
                if 'anio' in df.columns:
                    df = df[df['anio'] == año]
                elif 'fecha' in df.columns:
                    df = df[df['fecha'].dt.year == año]
            except ValueError:
                print(f"Periodo no numérico: {periodo}")

        # Filtrar por alcaldía
        alcaldia_encontrada = False
        if not todas_alcaldias and 'alcaldias' in df.columns:
            alcaldia_norm = normalize_text(alcaldia)

            # Filtrar por nombre normalizado de alcaldía
            alcaldias_dict = {
                "iztacalco": "IZTACALCO",
                "iztapalapa": "IZTAPALAPA",
                "alvaro obregon": "ALVARO OBREGON",
                "cuauhtemoc": "CUAUHTEMOC",
                "gustavo a madero": "GUSTAVO A MADERO",
                "tlahuac": "TLAHUAC",
                "tlalpan": "TLALPAN"
            }

            # Verificar cada clave conocida
            for key, value in alcaldias_dict.items():
                if key in alcaldia_norm:
                    filtered_df = df[df['alcaldias'].apply(
                        lambda x: normalize_text(x) == key if isinstance(x, str) else False
                    )]
                    if len(filtered_df) > 0:
                        df = filtered_df
                        alcaldia_para_mensaje = value
                        alcaldia_encontrada = True
                        break

            if not alcaldia_encontrada:
                # Intento adicional con todas las alcaldías disponibles
                alcaldias_unicas = df['alcaldias'].dropna().unique()
                alcaldias_unicas_norm = [normalize_text(alc) if isinstance(alc, str) else "" for alc in alcaldias_unicas]

                # Intento de coincidencia aproximada
                matches = difflib.get_close_matches(alcaldia_norm, alcaldias_unicas_norm, n=1, cutoff=0.8)
                if matches:
                    idx = alcaldias_unicas_norm.index(matches[0])
                    alcaldia_match = alcaldias_unicas[idx]
                    df = df[df['alcaldias'].apply(lambda x: normalize_text(x) == matches[0] if isinstance(x, str) else False)]
                    alcaldia_para_mensaje = alcaldia_match
                    alcaldia_encontrada = True

        # Contar delitos
        total_delitos = len(df)

        # Si se especificó una alcaldía pero no se encontró, dar un mensaje informativo
        if not todas_alcaldias and not alcaldia_encontrada:
            mensaje = f"No se encontraron registros de {tipo_delito_mensaje} para la alcaldía {alcaldia}"
            if periodo is not None:
                mensaje += f" durante {periodo}."
            else:
                mensaje += "."
            dispatcher.utter_message(text=mensaje)
            return []

        # Construir mensaje de respuesta
        if todas_alcaldias:
            mensaje = f"Se registraron {total_delitos} casos de {tipo_delito_mensaje} en toda la Ciudad de México"
        else:
            mensaje = f"Se registraron {total_delitos} casos de {tipo_delito_mensaje} en {alcaldia_para_mensaje}"

        # Agregar periodo al mensaje
        if periodo is not None:
            mensaje += f" durante {periodo}."
        else:
            mensaje += "."

        dispatcher.utter_message(text=mensaje)

        return []

endpoint

In [ ]:
# This file contains the different endpoints your bot can use.

# Server where the models are pulled from.
# https://rasa.com/docs/rasa/model-storage#fetching-models-from-a-server

#models:
#  url: http://my-server.com/models/default_core@latest
#  wait_time_between_pulls:  10   # [optional](default: 100)

# Server which runs your custom actions.
# https://rasa.com/docs/rasa/custom-actions

action_endpoint:
  url: "http://localhost:5055/webhook"

# Tracker store which is used to store the conversations.
# By default the conversations are stored in memory.
# https://rasa.com/docs/rasa/tracker-stores

#tracker_store:
#    type: redis
#    url: <host of the redis instance, e.g. localhost>
#    port: <port of your redis instance, usually 6379>
#    db: <number of your database within redis, e.g. 0>
#    password: <password used for authentication>
#    use_ssl: <whether or not the communication is encrypted, default false>

#tracker_store:
#    type: mongod
#    url: <url to your mongo instance, e.g. mongodb://localhost:27017>
#    db: <name of the db within your mongo instance, e.g. rasa>
#    username: <username used for authentication>
#    password: <password used for authentication>

# Event broker which all conversation events should be streamed to.
# https://rasa.com/docs/rasa/event-brokers

#event_broker:
#  url: localhost
#  username: username
#  password: password
#  queue: queue
